# exp_003 — Context × quantization interaction

This notebook analyzes measured trial artifacts only. It does not run a model or invent missing cells.

The required context × quantization and position × context views are descriptive and condition-limited.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'src').is_dir() and (candidate / 'experiments').is_dir()
)
sys.path.insert(0, str(ROOT / 'src'))
from llm_lab.analysis import (
    aggregate_jsonl,
    effective_context_by_variant_and_task,
    interaction_report,
    matched_cell_rows,
    relative_degradation_rows,
)

PHASE = 'main'
RESULTS_DIR = ROOT / 'experiments/exp_003-context_x_quantization/results'
RAW_PATH = RESULTS_DIR / 'raw' / f'{PHASE}-trials.jsonl'
SUMMARY_PATH = RESULTS_DIR / 'processed' / f'{PHASE}-summary.csv'
MANIFEST_PATH = RESULTS_DIR / 'manifests' / f'{PHASE}.json'
FINDINGS_PATH = ROOT / 'docs/findings.md'

for path in (RAW_PATH, SUMMARY_PATH, MANIFEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'measured exp_003 input is required: {path}')

run_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
source_manifest_path = Path(run_manifest['source_manifest'])
if not source_manifest_path.is_absolute():
    source_manifest_path = ROOT / source_manifest_path
if not source_manifest_path.is_file():
    raise FileNotFoundError(f'resolved source manifest is required: {source_manifest_path}')
source_sha256 = hashlib.sha256(source_manifest_path.read_bytes()).hexdigest()
if source_sha256 != run_manifest['source_manifest_sha256']:
    raise ValueError('source manifest SHA-256 does not match the run manifest')

summary = pd.read_csv(SUMMARY_PATH)
raw_summary = pd.DataFrame(aggregate_jsonl(RAW_PATH))
required_columns = {
    'variant_condition_id', 'task_type', 'target_context_tokens',
    'requested_evidence_position', 'context_instance_id', 'context_sha256',
    'scored_n', 'accuracy', 'attempted_n', 'end_to_end_success',
}
missing_columns = required_columns - set(summary.columns)
if missing_columns:
    raise ValueError(f'processed summary is missing required columns: {sorted(missing_columns)}')
if len(summary) != len(raw_summary):
    raise ValueError('processed summary row count does not match raw aggregation')
if any(row['status'] != 'valid' for row in run_manifest['coverage']):
    raise ValueError('exp_003 has excluded cells; complete matched-cell analysis is unavailable')

variant_ids = tuple(item['condition_id'] for item in run_manifest['quantization_variants'])
context_lengths = tuple(run_manifest['context_lengths'])
evidence_positions = tuple(run_manifest['evidence_positions'])
task_types = tuple(run_manifest['task_types'])
if run_manifest['planned_cell_n'] != len(variant_ids) * len(context_lengths) * len(evidence_positions) * len(task_types):
    raise ValueError('run manifest planned cell count does not match its dimensions')
matched = matched_cell_rows(
    summary.to_dict('records'),
    variant_ids=variant_ids,
    context_lengths=context_lengths,
    evidence_positions=evidence_positions,
    task_types=task_types,
)
matched_frame = pd.DataFrame(matched)
matched_frame.head()

In [ ]:
degradation = relative_degradation_rows(
    matched, baseline_context_tokens=min(context_lengths)
)
degradation_frame = pd.DataFrame(degradation)
degradation_frame[['variant_condition_id', 'target_context_tokens', 'accuracy_degradation', 'relative_degradation']]
gap_reports = interaction_report(
    matched,
    reference_variant='q8_0',
    approx_constant_gap_tolerance=0.10,
)
gap_report_frame = pd.DataFrame(gap_reports)
gap_report_frame[['task_type', 'variant_condition_id', 'classification', 'gap_change', 'matched_n' if 'matched_n' in gap_report_frame else 'classification']]

In [ ]:
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Context × quantization heatmaps, separated by task type.
for task_type in task_types:
    plot_frame = matched_frame[matched_frame['task_type'] == task_type]
    heatmap = plot_frame.groupby(['target_context_tokens', 'variant_condition_id'], as_index=False)['accuracy'].mean().pivot(index='target_context_tokens', columns='variant_condition_id', values='accuracy')
    figure, axis = plt.subplots(figsize=(7, 4))
    image = axis.imshow(heatmap.to_numpy(dtype=float), vmin=0, vmax=1, aspect='auto', cmap='viridis')
    axis.set_title(f'Context × quantization accuracy — {task_type}')
    axis.set_xlabel('Quantization variant')
    axis.set_ylabel('Context tokens')
    axis.set_xticks(range(len(heatmap.columns)), heatmap.columns)
    axis.set_yticks(range(len(heatmap.index)), heatmap.index)
    figure.colorbar(image, ax=axis, label='Accuracy')
    figure.tight_layout()
    figure.savefig(FIGURES_DIR / f'context-x-quantization-{task_type}.png', dpi=160)
    plt.show()

In [ ]:
# Position × context heatmaps, one for every quantization variant.
for variant_id in variant_ids:
    plot_frame = matched_frame[matched_frame['variant_condition_id'] == variant_id]
    heatmap = plot_frame.groupby(['target_context_tokens', 'requested_evidence_position'], as_index=False)['accuracy'].mean().pivot(index='target_context_tokens', columns='requested_evidence_position', values='accuracy')
    figure, axis = plt.subplots(figsize=(7, 4))
    image = axis.imshow(heatmap.to_numpy(dtype=float), vmin=0, vmax=1, aspect='auto', cmap='viridis')
    axis.set_title(f'Position × context accuracy — {variant_id}')
    axis.set_xlabel('Requested evidence position')
    axis.set_ylabel('Context tokens')
    axis.set_xticks(range(len(heatmap.columns)), [f'{value:.0%}' for value in heatmap.columns])
    axis.set_yticks(range(len(heatmap.index)), heatmap.index)
    figure.colorbar(image, ax=axis, label='Accuracy')
    figure.tight_layout()
    figure.savefig(FIGURES_DIR / f'position-x-context-{variant_id}.png', dpi=160)
    plt.show()

In [ ]:
# Quantization gap versus context, with matched scored-trial counts.
gap_points = []
for report in gap_reports:
    for point in report['context_points']:
        gap_points.append({
            'task_type': report['task_type'],
            'variant_condition_id': report['variant_condition_id'],
            **point,
        })
gap_points_frame = pd.DataFrame(gap_points)
figure, axis = plt.subplots(figsize=(8, 4))
for (task_type, variant_id), group in gap_points_frame.groupby(['task_type', 'variant_condition_id']):
    axis.plot(group['context_tokens'], group['quantization_gap'], marker='o', label=f'{task_type} / {variant_id}')
axis.set_title('Matched quantization gap versus context')
axis.set_xlabel('Context tokens')
axis.set_ylabel('Reference accuracy − variant accuracy')
axis.legend()
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'quantization-gap-vs-context.png', dpi=160)
plt.show()
gap_report_frame[['task_type', 'variant_condition_id', 'classification', 'shortest_context_gap', 'largest_context_gap', 'gap_change']]

In [ ]:
effective = effective_context_by_variant_and_task(
    matched,
    baseline_context_tokens=min(context_lengths),
    alpha=0.90,
    minimum_baseline_accuracy=0.80,
)
effective_frame = pd.DataFrame(effective)
effective_frame[['variant_condition_id', 'task_type', 'status', 'effective_context_tokens', 'crossing_context_tokens']]

# Findings are only eligible after this real manifest/table/figure audit.
findings_text = FINDINGS_PATH.read_text(encoding='utf-8')
if 'exp_003: not yet measured' in findings_text:
    print('exp_003 is not yet measured; no finding is added from this notebook run.')
else:
    print('Update docs/findings.md only with measured values, sample counts, manifest, table, figures, and caveats.')